In [1]:
# %pip install -qU pypdf
# %pip install -qU rapidocr-onnxruntime


In [2]:
import getpass
password = getpass.getpass("Senha para sudo: ")

# Instala o decodificador JBIG2
!echo {password} | sudo -S apt-get update
!echo {password} | sudo -S apt-get install -y jbig2dec

# Verifica se agora ele é encontrado
!jbig2dec --version

Senha para sudo:  ········


Obter:1 https://repo.steampowered.com/steam stable InRelease [3.622 B]
Atingido:2 https://download.docker.com/linux/ubuntu noble InRelease            
Atingido:3 https://packages.microsoft.com/repos/code stable InRelease          
Atingido:4 https://dl.google.com/linux/chrome/deb stable InRelease             
Atingido:5 http://security.ubuntu.com/ubuntu noble-security InRelease          
Atingido:6 http://archive.ubuntu.com/ubuntu noble InRelease                    
Atingido:7 http://archive.ubuntu.com/ubuntu noble-updates InRelease            
Atingido:8 http://archive.ubuntu.com/ubuntu noble-backports InRelease
Ignorar:9 https://ppa.launchpadcontent.net/alex-p/jbig2enc/ubuntu noble InRelease
Atingido:10 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu noble InRelease
Atingido:11 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu noble InRelease
Erro:12 https://ppa.launchpadcontent.net/alex-p/jbig2enc/ubuntu noble Release
  404  Not Found [IP: 2620:2d:4000:1::81 443]
L

In [7]:
import os
import warnings
from langchain_unstructured import UnstructuredLoader

# 1. Silenciar warnings de deprecação das bibliotecas internas (como o do max_size)
warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", message=".*max_size parameter is deprecated.*")

file_path = "./Data/PDF_C_Digitalizado.pdf"

# 2. Configuração Refatorada
loader = UnstructuredLoader(
    file_path=file_path,
    strategy="hi_res",
    
    # CORREÇÃO DO WARNING DE LÍNGUA:
    # Passamos português e inglês para garantir que o OCR funcione bem em ambos
    languages=["por", "eng", "deu"], 
    
    # CONFIGURAÇÕES DE EXTRAÇÃO:
    extract_images_in_pdf=True,
    infer_table_structure=True,
    
    # ESTRATÉGIA DE CHUNKING (Ajustada para o seu perfil técnico/psicologia)
    # Agrupar por título evita que conceitos sejam cortados ao meio
    chunking_strategy="by_title",
    max_characters=1000,      # Aumentei um pouco para manter o contexto
    new_after_n_chars=800,
    combine_text_under_n_chars=300,
)

# 3. Execução
docs = loader.load()

# 4. Visualização Limpa
print(f"\n[Sucesso] Processamento concluído: {len(docs)} elementos encontrados.")
print("-" * 50)

for doc in docs:
    # Com chunking, a categoria principal geralmente vira 'CompositeElement'
    category = doc.metadata.get("category")
    page = doc.metadata.get("page_number")
    
    # Vamos imprimir tudo para você ver como o Unstructured nomeou os blocos
    print(f"Página {page} | Tipo: {category}")
    print(f"Conteúdo: {doc.page_content.replace(chr(10), ' ')}...")
    print("-" * 20)




INFO: Reading PDF for file: ./Data/PDF_C_Digitalizado.pdf ...



[Sucesso] Processamento concluído: 13 elementos encontrados.
--------------------------------------------------
Página 1 | Tipo: CompositeElement
Conteúdo: DICIONÁBIO » PSICANALISE  ff  |  1.;All fMANN  DICIONÁRIO ENCICLOPÉDICO DE PSICANÁLISE  O legado de Freud e Lacan  editado por PIERRE KAUFMANN  Tradução: VERA RIBEIRO MARIA LUIZA X. DE A. BORGES  Supervisão da edição brasileira: MARCO ANTONIO COUTINHO JORGE psiquiatra, psicanalista  JORGE ZAHAR EDITOR Rio de Janeiro  facebook.com/lacanempdf  Título original:  L'apporl _fí-ewlien: É'limrnrs pour wze encrclopédic de la ps_Y(:Íluna/_\'Se  Tradução autorizada da edição francesa publicada cm l 993 por Éditions Bordas, de Paris. França  Copyright© 1993. Éditions Bordas  Copyright © l 996 da edição cm língua portuguesa:  Jorge Zahar Editor Ltda.  rua México 3 l sobreloja  20031-144 Rio de Janeiro, RJ  te!.: (21) 240-0226 / fax: (21) 262-5123  e-mail: jze(illzahar.com.br  Todos os direitos reservados....
--------------------
Página 3 | Tip

In [6]:
tables = [doc for doc in docs if doc.metadata.get("category") == "Table"]
print(f"Tabelas encontradas: {len(tables)}")


Tabelas encontradas: 0


In [7]:
from langchain_openai import ChatOpenAI
from langchain_openai import OpenAIEmbeddings

agent = ChatOpenAI(
    api_key="lm-studio",
    base_url= "http://localhost:1234/v1",
    model = "qwen/qwen3-4b-2507",
    temperature = 0 
    )


embeddings = OpenAIEmbeddings(
    base_url="http://localhost:1234/v1",
    api_key="lm-studio",
    model="nomic-embed-text-v1.5",
    check_embedding_ctx_length=False,  
    tiktoken_enabled=False,
)

In [8]:
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document

vectorstore = FAISS.from_documents(chunks, embeddings)
# Retriever (chama embeddings + similarity_search)
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

INFO: HTTP Request: POST http://localhost:1234/v1/embeddings "HTTP/1.1 200 OK"
INFO: Loading faiss with AVX2 support.
INFO: Successfully loaded faiss with AVX2 support.


In [9]:
from langchain_community.document_loaders import TextLoader

loader = TextLoader('./Data/PDF_C_Digitalizado_GT.txt', encoding="utf-8")
gt_a = loader.load()

gt_a[0].page_content

'Page 1:\n\nSIGMUND FREUD\nDICIONÁRIO ENCICLOPÉDICO DE PSICANÁLISE\nO legado de Freud e Lacan\neditado por PIERRE KAUFMANN\nTradução:\nVERA RIBEIRO\nMARIA LUIZA X. DE A. BORGES\nSupervisão da edição brasileira:\nMARCO ANTONIO COUTINHO JORGE\npsiquiatra, psicanalista\nJORGE ZAHAR EDITOR\nRio de Janeiro\n\nPage 2:\n\nTítulo original: L\'apport freudien: Éléments pour une encyclopédie de la psychanalyse\nTradução autorizada da edição francesa publicada em 1993 por Éditions Bordas, de Paris, França\nCopyright © 1993, Éditions Bordas\nCopyright © 1996 da edição em língua portuguesa: Jorge Zahar Editor Ltda.\nrua México 31 sobreloja\n20031-144 Rio de Janeiro, RJ\ntel.: (21) 240-0226 / fax: (21) 262-5123\ne-mail: jze@zahar.com.br\nTodos os direitos reservados.\nA reprodução não-autorizada desta publicação, no todo ou em parte, constitui violação do copyright (Lei 5.988)\nCapa: Carol Sá\nIlustração da capa: Infante, óleo sobre tela de Antonio Saura, 1960\n\nPage 3:\n\nCIP-Brasil. Catalogação-n

In [10]:
# 1. Corrigir template (use {gt_a} e remova /n)
from langchain_core.prompts import ChatPromptTemplate
prompt = ChatPromptTemplate.from_template("""
Você é um avaliador de qualidade de extração de dados.

Contexto extraído do PDF: {context}

Pergunta original: {question}

Ground Truth Correto: {gt_a}

Regra: As estruturas estão diferentes, não avaliar a estrutura, mas se o loader
(from langchain_community.document_loaders import PyPDFLoader)
é confiável e quanto.

Avalie:
1. Precisão: Quanto o RAG tem as mesmas palavras que o {gt_a}?
2. Recall: O contexto capturou todas as palavras do GT?
3. Score 0-10 e justificativa.

Resposta no formato:
Precisão: X/10
Recall: Y/10
Score Final: Z/10
Justificativa: ...
""")

# 2. Criar chain
rag_chain = prompt | agent  # Seu LLM (llm_arquivista ou agent)

# 3. Invocar com TODOS os inputs no dict
query = "Como avaliaria a qualidade da extração? Se baseando no ground_truth"
docs_relevantes = retriever.invoke(query)
context_str = "\n\n".join([doc.page_content for doc in docs_relevantes])


print("Context len:", len(context_str))
print("GT len:", len(gt_a))
print()

response = rag_chain.invoke({
    "context": context_str,
    "question": query,
    "gt_a": gt_a
})

print(response.content)

INFO: HTTP Request: POST http://localhost:1234/v1/embeddings "HTTP/1.1 200 OK"


Context len: 1909
GT len: 1



INFO: HTTP Request: POST http://localhost:1234/v1/chat/completions "HTTP/1.1 200 OK"


Precisão: 9/10  
Recall: 8/10  
Score Final: 8.5/10  
Justificativa: O contexto extraído apresenta uma correspondência muito próxima com o Ground Truth (GT), especialmente no que diz respeito ao conteúdo central sobre a "regra de abstinência", a sugestão oral de Freud sobre a histeria de angústia, a ideia de exposição ao afeto como mecanismo de superação da resistência, a definição da "técnica ativa" e a relação entre a proibição e a situação transferencial. As frases-chave — como "os pacientes, apesar de uma observância rigorosa da 'regra fundamental'", "ao se expor a esse afeto, os pacientes superam a resistência", "a partir de então, foi esse processo que pretendi designar pela expressão 'técnica ativa'", "a regra de abstinência deverá intervir desse duplo ponto de vista" — estão presentes com precisão e são expressas de forma coerente com o GT.

No entanto, há uma diferença de precisão em alguns detalhes de formatação e pontuação. O texto extraído contém erros de digitação e estrut

In [ ]:
# 8. MCP/Memory: Adicione checkpointer para persistir estado
from langgraph.checkpoint.memory import MemorySaver
checkpointer = MemorySaver()

# 4. State Schema
from typing_extensions import TypedDict, Annotated
import operator
from langchain_core.messages import AnyMessage

class AgentState(TypedDict):
    messages: Annotated[list[AnyMessage], operator.add]
    context: str  # Docs do PDF RAG
    research: str  # Arquivista output

# 3. Agent Prompts/Nodes (com RAG)
def arquivista_node(state: AgentState):
    # Use retriever aqui
    docs = retriever.invoke(state["messages"][-1].content)
    context = "\n".join([d.page_content for d in docs])
    
    prompt = f"""Arquivista: Analise com base no PDF: {context}\nQuery: {state["messages"][-1].content}"""
    response = llm_arquivista.invoke(prompt)
    return {"research": response.content, "messages": [response]}

def inspetor_node(state: AgentState):
    prompt = f"""Inspetor: Revise pesquisa: {state["research"]}\nPDF Context: {state["context"]}"""
    response = llm_inspetor.invoke(prompt)
    return {"messages": [response]}

# 5-6. Nodes + Edges
from langgraph.graph import StateGraph, START, END

workflow = StateGraph(AgentState)
workflow.add_node("arquivista", arquivista_node)
workflow.add_node("inspetor", inspetor_node)

workflow.add_edge(START, "arquivista")
workflow.add_edge("arquivista", "inspetor")
workflow.add_edge("inspetor", END)

graph = workflow.compile(checkpointer=checkpointer)

# Invocar com thread para memória
thread = {"configurable": {"thread_id": "1"}}
input = {"messages": [("user", "Pergunta sobre PDF")]}
for chunk in graph.stream(input, thread):
    print(chunk)

In [2]:
import warnings
from typing_extensions import TypedDict, Annotated
import operator
from langchain_core.messages import AnyMessage, HumanMessage, AIMessage
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.prompts import ChatPromptTemplate
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

# Silenciar warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

# ============================================================================
# 0. CONFIGURAÇÕES GLOBAIS (Atualizadas)
# ============================================================================
file_pdf = "./Data/PDF_C_Digitalizado.pdf"
file_gt = "./Data/PDF_C_Digitalizado_GT.txt"

llm_qwen = ChatOpenAI(  # Agente principal
    api_key="lm-studio", base_url="http://localhost:1234/v1",
    model="qwen/qwen3-4b-2507", temperature=0
)

embeddings = OpenAIEmbeddings(
    base_url="http://localhost:1234/v1", api_key="lm-studio",
    model="nomic-embed-text-v1.5",
    check_embedding_ctx_length=False, tiktoken_enabled=False
)

# Load PDF e GT
from langchain_unstructured import UnstructuredLoader
from langchain_community.document_loaders import TextLoader

loader_pdf = UnstructuredLoader(file_pdf, strategy="hi_res", languages=["por", "eng"])
docs_pdf = loader_pdf.load()
chunks = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200).split_documents(docs_pdf)

loader_gt = TextLoader(file_gt, encoding="utf-8")
gt_docs = loader_gt.load()
GT_CONTENT = gt_docs[0].page_content  # Ground truth como string única

# Vectorstore RAG
from langchain_community.vectorstores import FAISS
vectorstore = FAISS.from_documents(chunks, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

print(f"✅ PDF: {len(chunks)} chunks | GT carregado: {len(GT_CONTENT)} chars")

# ============================================================================
# 1. STATE SCHEMA (DDD)
# ============================================================================
class AvaliacaoState(TypedDict):
    messages: Annotated[list[AnyMessage], operator.add]
    context_rag: str          # Contexto extraído do PDF via RAG
    ground_truth: str         # GT fixo
    avaliacao: str            # Output do avaliador

# ============================================================================
# 2. AGENTE 1: GT_AGENT (Carrega Ground Truth)
# ============================================================================
def gt_agent(state: AvaliacaoState) -> dict:
    """Agente que fornece o ground-truth para comparação."""
    query = state["messages"][-1].content
    
    return {
        "ground_truth": GT_CONTENT,
        "messages": [AIMessage(content=f"🔍 GT carregado ({len(GT_CONTENT)} chars): {GT_CONTENT[:200]}...")]
    }

# ============================================================================
# 3. AGENTE 2: AVALIADOR_AGENT (Realiza avaliação com RAG + GT)
# ============================================================================
avaliador_prompt = ChatPromptTemplate.from_template("""
Você é um avaliador de qualidade de extração de dados de PDF.

CONTEXTO EXTRAÍDO (RAG): {context_rag}

PERGUNTA: {question}

GROUND-TRUTH (EXTRAÇÃO HUMANA PERFEITA): {ground_truth}

Tarefa: Avalie se o loader (UnstructuredLoader hi_res) é confiável.
- **Ignore diferenças de estrutura/formatação** (foco no conteúdo)
- Compare **palavras-chave e informações principais**

Formato EXATO:

Precisão: X/10 (quantas palavras/infos do GT estão no contexto?)
Recall: Y/10 (o contexto capturou tudo do GT?)
Score Final: Z/10 Justificativa: [análise detalhada]

""")

def avaliador_agent(state: AvaliacaoState) -> dict:
    """Agente avaliador com RAG + GT."""
    query = state["messages"][-1].content
    
    # RAG: Buscar contexto relevante do PDF
    docs_relevantes = retriever.invoke(query)
    context_rag = "\n\n".join([doc.page_content for doc in docs_relevantes])
    
    # Chain com prompt + LLM
    chain = avaliador_prompt | llm_qwen
    response = chain.invoke({
        "context_rag": context_rag,
        "question": query,
        "ground_truth": state["ground_truth"][:4000]  # Limite contexto LLM
    })
    
    return {
        "context_rag": context_rag,
        "avaliacao": response.content,
        "messages": [AIMessage(content=f"📊 Avaliação: {response.content}")]
    }

# ============================================================================
# 4-6. WORKFLOW: Nodes + Edges (A2A)
# ============================================================================
workflow = StateGraph(AvaliacaoState)

# Add nodes
workflow.add_node("gt_agent", gt_agent)
workflow.add_node("avaliador_agent", avaliador_agent)

# Edges: GT → Avaliador → END
workflow.add_edge(START, "gt_agent")
workflow.add_edge("gt_agent", "avaliador_agent")
workflow.add_edge("avaliador_agent", END)

# ============================================================================
# 8. COMPILAR COM MEMORY (MCP)
# ============================================================================
checkpointer = MemorySaver()
app = workflow.compile(checkpointer=checkpointer)

# ============================================================================
# USAR: Com thread para memória persistente
# ============================================================================
thread_config = {"configurable": {"thread_id": "avaliacao_pdf_c"}}
query = "Avalie a extração de tabelas e texto do PDF"

for event in app.stream({"messages": [HumanMessage(content=query)]}, thread_config, stream_mode="values"):
    event["messages"][-1].pretty_print()


INFO: Reading PDF for file: ./Data/PDF_C_Digitalizado.pdf ...
INFO: HTTP Request: POST http://localhost:1234/v1/embeddings "HTTP/1.1 200 OK"
INFO: Loading faiss with AVX2 support.
INFO: Successfully loaded faiss with AVX2 support.
INFO: HTTP Request: POST http://localhost:1234/v1/embeddings "HTTP/1.1 200 OK"


✅ PDF: 72 chunks | GT carregado: 7834 chars
================================ Human Message =================================

Avalie a extração de tabelas e texto do PDF
================================== Ai Message ==================================

🔍 GT carregado (7834 chars): Page 1:

SIGMUND FREUD
DICIONÁRIO ENCICLOPÉDICO DE PSICANÁLISE
O legado de Freud e Lacan
editado por PIERRE KAUFMANN
Tradução:
VERA RIBEIRO
MARIA LUIZA X. DE A. BORGES
Supervisão da edição brasileira:...


INFO: HTTP Request: POST http://localhost:1234/v1/chat/completions "HTTP/1.1 200 OK"


================================== Ai Message ==================================

📊 Avaliação: **Avaliação da confiabilidade do loader (UnstructuredLoader hi_res)**  
*(Baseado no contexto extraído e na Ground-Truth – GT – de extração humana perfeita)*

---

### ✅ **Formato EXATO:**

- **Precisão:** 8/10  
- **Recall:** 7/10  
- **Score Final:** **7.5/10**

---

### 🔍 **Justificativa detalhada:**

#### ✅ **Precisão (8/10):**  
O loader captura **a maioria das palavras-chave e informações principais** do GT, com alta fidelidade ao conteúdo essencial.

- **Título do livro:**  
  GT: *"Dicionário enciclopédico de psicanálise: o legado de Freud e Lacan"*  
  Contexto: "D542 Dicionário enciclopédico de psicanálise: o legado de Freud e Lacan" → **correta e completa**

- **Autores/editores:**  
  GT: *editado por Pierre Kaufmann; tradução, Vera Ribeiro, Maria Luiza X. de A. Borges*  
  Contexto: "editado por Pierre Kaufrnann: tradução, Vera Ribeiro. Maria Luiza X. de A. Borges: consultoria." 

In [8]:
curl -X POST http://localhost:8000/process \
  -H "Content-Type: application/json" \
  -d '{"text": "Olá mundo Docker"}'

SyntaxError: invalid syntax (2258939124.py, line 1)